In [ ]:
!pip install --upgrade git+https://github.com/akashskypatel/NeurCross.git
!pip install --upgrade trimesh huggingface_hub kagglehub torchinfo

In [ ]:
from google.colab import userdata
import os

HF_DATASET_REPO_ID = "akashskypatel/NeurCross-TEST"
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip()
    raise RuntimeError(
        "Missing Colab secret HF_TOKEN. Add it from the Colab Secrets panel "
        "and grant this notebook access."
    )

os.environ["HF_TOKEN"] = HF_TOKEN

KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY = userdata.get("KAGGLE_KEY")
if KAGGLE_USERNAME:
    KAGGLE_USERNAME = KAGGLE_USERNAME.strip()
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:
    KAGGLE_KEY = KAGGLE_KEY.strip()
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

print(f"HF dataset target: {HF_DATASET_REPO_ID}")
print("Kaggle credentials detected from Colab secrets." if KAGGLE_USERNAME and KAGGLE_KEY else "Kaggle credentials not found in Colab secrets; kagglehub may still use an existing session.")

In [ ]:
from pathlib import Path

KAGGLE_DATASET_IDS = [
    "balraj98/modelnet40-princeton-3d-object-dataset",
    "mrbiid/meshy-ai-800-glb-3d-assets-categorised-and-labelled",
    "programmer3/cadcae-design-dataset",
    "timfuger/part-processing-dataset",
    "chienru/mask2-3d",
    "sc0v1n0/3d-models",
    "lukaszfuszara/thingi10k-name-and-category",
    "amytai/nutritionverse-3d",
    "mrbiid/meshy-ai-363-ply-creatures-labelled",
]

DATA_SOURCES = [{"kind": "kaggle", "id": dataset_id} for dataset_id in KAGGLE_DATASET_IDS]

ALLOWED_EXTENSIONS = {
    ".obj",
    ".stl",
    ".off",
    ".ply",
    ".dae",
    ".collada",
    ".json",
    ".dict",
    ".glb",
    ".dict64",
    ".msgpack",
}

RUNTIME_ROOT = Path("/content/neurcross_colab")
DOWNLOAD_ROOT = RUNTIME_ROOT / "downloads"
RUN_ROOT = RUNTIME_ROOT / "runs"
LABEL_ROOT = RUN_ROOT / "dataset_labels"
UPLOAD_ROOT = RUNTIME_ROOT / "hf_uploads"
TRACKER_LOCAL = RUNTIME_ROOT / "training_tracker.jsonl"
STATE_LOCAL = RUNTIME_ROOT / "training_state.json"

HF_TRACKER_PATH = "metadata/training_tracker.jsonl"
HF_STATE_PATH = "metadata/training_state.json"
HF_OUTPUT_PREFIX = "generated-labels"

MAX_MESH_RUNS_PER_EXECUTION = 1
RETRY_FAILED_RUNS = True
RETRY_INTERRUPTED_RUNS = True
DELETE_DOWNLOADED_SOURCE_AFTER_USE = True
DELETE_SOURCE_MESH_AFTER_SUCCESS = True
CREATE_HF_PR = False

DEFAULT_TRAINING_OVERRIDES = {
    "device": "auto",
    "total_steps": 10000,
    "steps_per_epoch": 1000,
    "n_points": 15000,
    "nonmnfld_sample_type": "mixed",
    "save_best_by": "val_field_score",
    "early_stop": True,
    "early_stop_min_steps": 2000,
    "early_stop_patience": 1000,
    "early_stop_smooth_window": 100,
    "export_sdf_samples": True,
    "export_geometry_npz": True,
    "export_features": True,
    "quality_gate": "default",
    "num_workers": 0,
    "keep_last_n_checkpoints": 1,
    "max_topology_memory_gb": 8.0,
}

for path in (DOWNLOAD_ROOT, RUN_ROOT, LABEL_ROOT, UPLOAD_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(f"Configured {len(DATA_SOURCES)} data sources.")
print(f"Allowed mesh-like extensions: {sorted(ALLOWED_EXTENSIONS)}")
print("Effective generate-label defaults:")
for key, value in DEFAULT_TRAINING_OVERRIDES.items():
    print(f"  {key} = {value}")

In [ ]:
import gc
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import EntryNotFoundError, RepositoryNotFoundError
import kagglehub

NEURCROSS_MODULE = "neurcross"
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=HF_DATASET_REPO_ID, repo_type="dataset", exist_ok=True)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def safe_name(value: str, fallback: str = "unnamed") -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip(".-_")
    return value[:128] or fallback


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def read_json(path: Path, default):
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)
        handle.write("")


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True, separators=(",", ":")))
            handle.write("")


def load_remote_jsonl(filename: str) -> list[dict]:
    try:
        local_path = hf_hub_download(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            filename=filename,
            token=os.environ["HF_TOKEN"],
        )
        return read_jsonl(Path(local_path))
    except (EntryNotFoundError, RepositoryNotFoundError, FileNotFoundError):
        return []


def load_remote_json(filename: str, default):
    try:
        local_path = hf_hub_download(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            filename=filename,
            token=os.environ["HF_TOKEN"],
        )
        return read_json(Path(local_path), default)
    except (EntryNotFoundError, RepositoryNotFoundError, FileNotFoundError):
        return default


def latest_tracker_map(records: list[dict]) -> dict[str, dict]:
    latest = {}
    for record in records:
        mesh_key = record.get("mesh_key")
        if not mesh_key:
            continue
        current = latest.get(mesh_key)
        if current is None or record.get("updated_at_utc", "") >= current.get("updated_at_utc", ""):
            latest[mesh_key] = record
    return latest


def load_tracker_records() -> list[dict]:
    merged = latest_tracker_map(load_remote_jsonl(HF_TRACKER_PATH) + read_jsonl(TRACKER_LOCAL))
    return sorted(merged.values(), key=lambda item: (item.get("source_id", ""), item.get("mesh_relative_path", ""), item.get("updated_at_utc", "")))


def default_state() -> dict:
    return {
        "updated_at_utc": utc_now(),
        "last_source_id": None,
        "last_mesh_key": None,
        "last_status": None,
        "runs_completed": 0,
        "source_status": {},
    }


def load_state() -> dict:
    state = default_state()
    state.update(load_remote_json(HF_STATE_PATH, {}))
    state.update(read_json(STATE_LOCAL, {}))
    state.setdefault("source_status", {})
    return state


def upload_progress(tracker_records: list[dict], state: dict, *, create_pr: bool = False) -> None:
    tracker_records = sorted(
        latest_tracker_map(tracker_records).values(),
        key=lambda item: (item.get("source_id", ""), item.get("mesh_relative_path", ""), item.get("updated_at_utc", "")),
    )
    state = dict(state)
    state["updated_at_utc"] = utc_now()
    write_jsonl(TRACKER_LOCAL, tracker_records)
    write_json(STATE_LOCAL, state)
    api.upload_file(
        path_or_fileobj=str(TRACKER_LOCAL),
        path_in_repo=HF_TRACKER_PATH,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Update NeurCross training tracker",
        create_pr=create_pr,
    )
    api.upload_file(
        path_or_fileobj=str(STATE_LOCAL),
        path_in_repo=HF_STATE_PATH,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Update NeurCross training state",
        create_pr=create_pr,
    )


def source_id(source: dict) -> str:
    return f"{source['kind']}:{source['id']}"


def source_slug(source: dict) -> str:
    return safe_name(source_id(source))


def build_mesh_key(source: dict, relative_path: str) -> str:
    return f"{source_id(source)}::{relative_path.replace(os.sep, '/')}"


def mesh_record_status(record: dict | None) -> str | None:
    if not record:
        return None
    return record.get("status")


def should_process_mesh(record: dict | None) -> bool:
    status = mesh_record_status(record)
    if status == "completed":
        return False
    if status == "failed":
        return RETRY_FAILED_RUNS
    if status == "started":
        return RETRY_INTERRUPTED_RUNS
    if status == "skipped":
        return RETRY_FAILED_RUNS
    return True


def download_source_dataset(source: dict) -> Path:
    if source["kind"] != "kaggle":
        raise ValueError(f"Unsupported source kind: {source['kind']}")
    local_path = Path(kagglehub.dataset_download(source["id"]))
    print(f"Downloaded {source_id(source)} -> {local_path}")
    return local_path


def discover_mesh_entries(source: dict, local_root: Path) -> list[dict]:
    entries = []
    for path in sorted(local_root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue
        relative_path = path.relative_to(local_root).as_posix()
        entries.append(
            {
                "source": source,
                "source_id": source_id(source),
                "local_root": local_root,
                "mesh_path": path,
                "mesh_relative_path": relative_path,
                "mesh_filename": path.name,
                "mesh_key": build_mesh_key(source, relative_path),
            }
        )
    return entries


def source_pending_entries(mesh_entries: list[dict], tracker_records: list[dict]) -> list[dict]:
    tracker_map = latest_tracker_map(tracker_records)
    return [entry for entry in mesh_entries if should_process_mesh(tracker_map.get(entry["mesh_key"]))]


def build_sample_id(mesh_entry: dict) -> str:
    base = safe_name(Path(mesh_entry["mesh_relative_path"]).stem, "mesh")
    suffix = hashlib.sha1(mesh_entry["mesh_key"].encode("utf-8")).hexdigest()[:12]
    return f"{base}-{suffix}"


def cli_args_from_kwargs(kwargs: dict) -> list[str]:
    args = []
    for key, value in kwargs.items():
        flag = f"--{key}"
        if value is None:
            continue
        if isinstance(value, bool):
            if value:
                args.append(flag)
            continue
        if isinstance(value, (list, tuple)):
            args.append(flag)
            args.extend(str(item) for item in value)
            continue
        args.extend([flag, str(value)])
    return args


def run_subprocess_streaming(cmd: list[str]) -> None:
    """Run neurcross in a child process and mirror stdout/stderr live into the notebook."""
    resolved_env = os.environ.copy()
    resolved_env["PYTHONUNBUFFERED"] = "1"
    print("Running command:", flush=True)
    print(" ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=resolved_env,
    )
    assert process.stdout is not None
    for line in iter(process.stdout.readline, ""):
        if not line:
            break
        print(line, end="", flush=True)
    process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


def find_sample_dir(dataset_root: Path, sample_id: str) -> Path | None:
    candidates = [
        dataset_root / "accepted" / sample_id,
        dataset_root / "quarantine" / sample_id,
        dataset_root / "failed" / sample_id,
        dataset_root / sample_id,
    ]
    for candidate in candidates:
        if (candidate / "manifest.json").exists():
            return candidate
    return None


def upload_sample_dir(sample_dir: Path, source: dict, sample_id: str, *, create_pr: bool = False) -> str:
    path_in_repo = f"{HF_OUTPUT_PREFIX}/{datetime.now(timezone.utc):%Y/%m/%d}/{source_slug(source)}/{sample_id}"
    commit = api.upload_folder(
        folder_path=str(sample_dir),
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        path_in_repo=path_in_repo,
        commit_message=f"Add NeurCross sample {sample_id}",
        create_pr=create_pr,
    )
    print(f"Uploaded {sample_id} -> {HF_DATASET_REPO_ID}/{path_in_repo}")
    return path_in_repo


def cleanup_runtime(*, sample_dir: Path | None = None, source_root: Path | None = None, remove_mesh_path: Path | None = None) -> None:
    if sample_dir and sample_dir.exists():
        shutil.rmtree(sample_dir, ignore_errors=True)
    if remove_mesh_path and DELETE_SOURCE_MESH_AFTER_SUCCESS and remove_mesh_path.exists():
        try:
            remove_mesh_path.unlink()
        except OSError as exc:
            print(f"Could not delete source mesh {remove_mesh_path}: {exc}")
    if source_root and DELETE_DOWNLOADED_SOURCE_AFTER_USE and source_root.exists():
        shutil.rmtree(source_root, ignore_errors=True)
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()


def mesh_file_info(mesh_entry: dict) -> dict:
    mesh_path = mesh_entry["mesh_path"]
    stat = mesh_path.stat()
    return {
        "mesh_sha256": sha256_file(mesh_path),
        "mesh_size_bytes": int(stat.st_size),
        "mesh_suffix": mesh_path.suffix.lower(),
    }


def update_tracker_record(records: list[dict], mesh_key: str, payload: dict) -> list[dict]:
    latest = latest_tracker_map(records)
    latest[mesh_key] = payload
    return list(latest.values())

In [ ]:
def process_mesh_entry(mesh_entry: dict, tracker_records: list[dict], state: dict) -> tuple[list[dict], dict, bool]:
    sample_id = build_sample_id(mesh_entry)
    sample_dataset_root = LABEL_ROOT
    mesh_info = mesh_file_info(mesh_entry)
    command_kwargs = {
        "data_path": str(mesh_entry["mesh_path"]),
        "dataset_root": str(sample_dataset_root),
        "sample_id": sample_id,
        "overwrite": True,
        **DEFAULT_TRAINING_OVERRIDES,
    }
    command = [sys.executable, "-u", "-m", NEURCROSS_MODULE, "generate-label", *cli_args_from_kwargs(command_kwargs)]
    mesh_key = mesh_entry["mesh_key"]
    previous = latest_tracker_map(tracker_records).get(mesh_key, {})
    started_at = utc_now()
    state.update(
        {
            "last_source_id": mesh_entry["source_id"],
            "last_mesh_key": mesh_key,
            "last_status": "started",
        }
    )
    tracker_records = update_tracker_record(
        tracker_records,
        mesh_key,
        {
            "mesh_key": mesh_key,
            "source_id": mesh_entry["source_id"],
            "source_kind": mesh_entry["source"]["kind"],
            "dataset_id": mesh_entry["source"]["id"],
            "mesh_relative_path": mesh_entry["mesh_relative_path"],
            "mesh_filename": mesh_entry["mesh_filename"],
            "mesh_sha256": mesh_info["mesh_sha256"],
            "mesh_size_bytes": mesh_info["mesh_size_bytes"],
            "mesh_suffix": mesh_info["mesh_suffix"],
            "sample_id": sample_id,
            "status": "started",
            "attempt": int(previous.get("attempt", 0)) + 1,
            "updated_at_utc": started_at,
            "started_at_utc": started_at,
            "error": None,
            "hf_path": previous.get("hf_path"),
            "command": " ".join(command),
        },
    )
    upload_progress(tracker_records, state, create_pr=CREATE_HF_PR)

    sample_dir = None
    manifest = None
    uploaded_to_hf = False
    path_in_repo = None
    final_status = "failed"
    error_message = None

    try:
        run_subprocess_streaming(command)
    except Exception as exc:
        error_message = repr(exc)
        print(f"Training command failed for {mesh_entry['mesh_relative_path']}: {error_message}")
    finally:
        sample_dir = find_sample_dir(sample_dataset_root, sample_id)
        if sample_dir and (sample_dir / "manifest.json").exists():
            with (sample_dir / "manifest.json").open("r", encoding="utf-8") as handle:
                manifest = json.load(handle)

    if sample_dir and manifest is not None:
        path_in_repo = upload_sample_dir(sample_dir, mesh_entry["source"], sample_id, create_pr=CREATE_HF_PR)
        uploaded_to_hf = True
        final_status = manifest.get("sample_state", "completed")
        quality = manifest.get("quality", {})
        recommended_destination = quality.get("recommended_destination")
    else:
        recommended_destination = None

    finished_at = utc_now()
    tracker_records = update_tracker_record(
        tracker_records,
        mesh_key,
        {
            "mesh_key": mesh_key,
            "source_id": mesh_entry["source_id"],
            "source_kind": mesh_entry["source"]["kind"],
            "dataset_id": mesh_entry["source"]["id"],
            "mesh_relative_path": mesh_entry["mesh_relative_path"],
            "mesh_filename": mesh_entry["mesh_filename"],
            "mesh_sha256": mesh_info["mesh_sha256"],
            "mesh_size_bytes": mesh_info["mesh_size_bytes"],
            "mesh_suffix": mesh_info["mesh_suffix"],
            "sample_id": sample_id,
            "status": final_status,
            "attempt": int(previous.get("attempt", 0)) + 1,
            "updated_at_utc": finished_at,
            "started_at_utc": started_at,
            "finished_at_utc": finished_at,
            "error": error_message,
            "hf_path": path_in_repo,
            "uploaded_to_hf": uploaded_to_hf,
            "recommended_destination": recommended_destination,
            "command": " ".join(command),
        },
    )
    state.update(
        {
            "last_source_id": mesh_entry["source_id"],
            "last_mesh_key": mesh_key,
            "last_status": final_status,
            "runs_completed": int(state.get("runs_completed", 0)) + (1 if final_status == "completed" else 0),
        }
    )
    upload_progress(tracker_records, state, create_pr=CREATE_HF_PR)

    cleanup_runtime(
        sample_dir=sample_dir,
        source_root=None,
        remove_mesh_path=mesh_entry["mesh_path"],
    )

    return tracker_records, state, final_status == "completed"


def process_sources(max_mesh_runs: int = MAX_MESH_RUNS_PER_EXECUTION) -> None:
    tracker_records = load_tracker_records()
    state = load_state()
    processed_this_execution = 0

    for source in DATA_SOURCES:
        if processed_this_execution >= max_mesh_runs:
            break

        sid = source_id(source)
        if state.get("source_status", {}).get(sid) == "complete":
            print(f"Skipping completed source: {sid}")
            continue

        local_root = download_source_dataset(source)
        try:
            mesh_entries = discover_mesh_entries(source, local_root)
            print(f"Discovered {len(mesh_entries)} candidate files in {sid}")
            pending_entries = source_pending_entries(mesh_entries, tracker_records)

            if not pending_entries:
                print(f"No pending meshes remain for {sid}")
                state.setdefault("source_status", {})[sid] = "complete"
                state["last_source_id"] = sid
                state["last_status"] = "source_complete"
                upload_progress(tracker_records, state, create_pr=CREATE_HF_PR)
                continue

            for mesh_entry in pending_entries:
                print(f"Processing {mesh_entry['mesh_relative_path']} from {sid}")
                tracker_records, state, _ = process_mesh_entry(mesh_entry, tracker_records, state)
                processed_this_execution += 1
                if processed_this_execution >= max_mesh_runs:
                    break

            remaining_entries = source_pending_entries(mesh_entries, tracker_records)
            if not remaining_entries:
                state.setdefault("source_status", {})[sid] = "complete"
                state["last_source_id"] = sid
                state["last_status"] = "source_complete"
                upload_progress(tracker_records, state, create_pr=CREATE_HF_PR)
                print(f"Marked {sid} complete in HF state.")
        finally:
            cleanup_runtime(source_root=local_root)

    print(f"Processed {processed_this_execution} mesh run(s) in this execution.")
    state = load_state()
    print(json.dumps(state, indent=2, sort_keys=True))

In [ ]:
# Execute one or more resumable per-mesh generate-label runs.
# Keep MAX_MESH_RUNS_PER_EXECUTION low in Colab so each execution stays stable.
process_sources()

## Training Flow

```mermaid
flowchart TD
    A[Load Colab secrets and config] --> B[Load HF tracker and resume state]
    B --> C{More mesh runs allowed this execution?}
    C -- No --> Z[Stop and print final state]
    C -- Yes --> D[Select next Kaggle dataset source]
    D --> E[Download one source dataset locally]
    E --> F[Discover supported mesh files]
    F --> G[Filter meshes against HF tracker]
    G --> H{Pending mesh exists?}
    H -- No --> S{Source still has pending meshes?}
    S -- Yes --> G
    S -- No --> I[Mark the source complete]
    I --> J[Delete downloaded source dataset]
    J --> C
    H -- Yes --> K[Record started status in HF tracker]
    K --> L[Run python -u -m neurcross generate-label in subprocess]
    L --> N{Manifest/sample dir produced?}
    N -- Yes --> O[Upload per-mesh label package to HF dataset]
    O --> P[Update HF tracker with completed or skipped status]
    N -- No --> Q[Update HF tracker with failed status]
    P --> R[Clean sample dir, CUDA cache, and optional source mesh]
    Q --> R
    R --> S{Source still has pending meshes?}
```
